# Lab · Fine-tune on a platform — Kubeflow Trainer & Amazon SageMaker

## What this is

The **same fine-tune as Kaggle Lab 2** — teach `Qwen2.5-3B-Instruct` to turn a
free-text order message into a **strict JSON object** with QLoRA — but written in
the *language of the two reference platforms* from the README's
[Systematic view](../../README.md#systematic-view--kubeflow---sagemaker):
**Kubeflow Trainer** (open, on your Kubernetes) and **Amazon SageMaker** (managed).

Lab 2 ran the whole thing inline in one Kaggle cell. A platform splits the same
work into **a training entrypoint** + **a job you submit** + **a registered model**
+ **an endpoint**. The training code does not change — only how you *launch* it.

| Lifecycle stage | Kubeflow | AWS (SageMaker) | this notebook |
|---|---|---|---|
| Model Training & fine-tuning | **Kubeflow Trainer** | **SageMaker Training Jobs** | Tracks A / B |
| ML metadata + artifacts | **Kubeflow Hub** | **SageMaker Model Registry** | register step |
| Model Serving | **KServe** *(ecosystem)* | **SageMaker Endpoints** | deploy step |
| Orchestrate | **Kubeflow Pipelines** | **SageMaker Pipelines** | see README |
| Run everything | **Kubernetes** | **Amazon EKS / EC2** | the cluster/account |

> **This is a reference lab, not a free-Kaggle one.** It needs a Kubeflow cluster
> *or* an AWS account with SageMaker — both cost money and have setup. The
> *runnable, free* version of this exact fine-tune is
> [`../cloud-kaggle/lab2_finetune_kaggle.ipynb`](../cloud-kaggle/lab2_finetune_kaggle.ipynb).
> Run the cells on the platform you have; read the other track to compare idioms.

The shared training logic is in [`train.py`](train.py) — the *entrypoint* both
platforms invoke. Open it alongside this notebook; the cells below only **submit**
it.

---
## Track A · Kubeflow Trainer

Kubeflow's idiom is **a Python function** handed to a `CustomTrainer`. The Kubeflow
SDK packages the function, runs it as a distributed **TrainJob** on a chosen
*runtime* (a base image + framework), and streams logs back. You run these cells
from a **Kubeflow Notebook** inside the cluster (which carries the in-cluster
credentials).

### A1 — connect and pick a runtime

`list_runtimes()` shows the framework images the cluster offers (e.g. a
`torch-distributed` runtime). The `TrainerClient` uses your in-cluster service
account — no kubeconfig wrangling from a Kubeflow Notebook.

In [ ]:
# pip install kubeflow      # the unified Kubeflow SDK (kubeflow.trainer)
from kubeflow.trainer import TrainerClient, CustomTrainer, Runtime

client = TrainerClient()
for r in client.list_runtimes():
    print(r.name)

runtime = client.get_runtime('torch-distributed')   # a PyTorch runtime image

### A2 — the training function

This is the **same body as [`train.py`](train.py)**, written as a self-contained
function so the SDK can ship it to the cluster. `packages_to_install` adds the
fine-tuning deps on top of the runtime image. In production you'd instead bake
`train.py` into the runtime image and call `from train import train` — keeping a
single source of truth — but inlining keeps this cell runnable on any runtime.

In [ ]:
def order_parser_train():
    import os
    # one source of truth: the deps image carries train.py, or fetch it here.
    # For the workshop we install deps and import the shared entrypoint module.
    os.environ.setdefault('MLFLOW_TRACKING_URI', 'http://mlflow.kubeflow:5000')  # in-cluster tracking
    import argparse
    from train import train            # train.py baked into the runtime image
    args = argparse.Namespace(
        model_name='Qwen/Qwen2.5-3B-Instruct',
        epochs=1, lr=2e-4, lora_r=16, lora_alpha=32,
        train_dir='/workspace/data',   # an emptyDir / PVC the func can write to
        model_dir='/workspace/model')  # adapter lands here; copy to Kubeflow Hub after
    train(args)

### A3 — submit the TrainJob and stream logs

`TrainerClient().train(...)` returns a job id. One GPU node is plenty for a 3B
QLoRA run; scale `num_nodes` for larger models. `get_job` reports per-step status;
`get_job_logs(follow=True)` tails the loss curve live.

In [ ]:
job_id = TrainerClient().train(
    runtime=runtime,
    trainer=CustomTrainer(
        func=order_parser_train,
        num_nodes=1,
        resources_per_node={'cpu': 4, 'memory': '24Gi', 'gpu': 1},
        packages_to_install=['trl', 'peft', 'bitsandbytes', 'datasets', 'accelerate', 'mlflow'],
    ),
)
print('submitted TrainJob:', job_id)

for s in TrainerClient().get_job(name=job_id).steps:
    print('step', s.name, '->', s.status)

for line in TrainerClient().get_job_logs(job_id, follow=True):
    print(line)

### A4 — register + serve (Kubeflow)

After the job, the adapter sits in the job's volume. Push it to **Kubeflow Hub**
(ML metadata + artifacts) so the run is versioned and discoverable, then serve it
with **KServe** — an `InferenceService` is the Kubeflow-ecosystem equivalent of a
SageMaker Endpoint.

```yaml
# kserve-order-parser.yaml  —  apply with: kubectl apply -f kserve-order-parser.yaml
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: order-parser
spec:
  predictor:
    model:
      modelFormat: { name: huggingface }
      args: ['--model_id=Qwen/Qwen2.5-3B-Instruct', '--model_dir=/mnt/models']
      storageUri: 'pvc://order-parser-model/'   # or an s3:// / hub:// artifact uri
      resources: { limits: { nvidia.com/gpu: '1' } }
```

---
## Track B · Amazon SageMaker

SageMaker's idiom is **a script** (`entry_point`) wrapped in an *Estimator*.
SageMaker spins up a managed training instance, runs `python train.py` with your
`hyperparameters` as CLI flags, mounts the `train` channel at `SM_CHANNEL_TRAIN`,
and uploads whatever lands in `SM_MODEL_DIR` to S3 as `model.tar.gz`. Run these
cells from **SageMaker Studio** (or anywhere with AWS creds + the SDK).

### B1 — session and role

In [ ]:
# pip install 'sagemaker>=2.220'
import sagemaker
from sagemaker.huggingface import HuggingFace

sess = sagemaker.Session()
role = sagemaker.get_execution_role()      # the Studio execution role
region = sess.boto_region_name
print('region:', region, '| bucket:', sess.default_bucket())

### B2 — the Training Job

`entry_point='train.py'` is the **same file** Kubeflow's func imports.
`hyperparameters` are delivered to it as `--epochs`, `--lora_r`, … exactly as its
`argparse` expects. Pick a current Hugging Face Deep Learning Container by setting
the framework versions; `ml.g5.2xlarge` (one A10G, 24 GB) comfortably fits the 3B
QLoRA run. `train.py` synthesizes the dataset itself, so no `train` channel is
required — pass `inputs={'train': s3_uri}` to use your own.

In [ ]:
estimator = HuggingFace(
    entry_point='train.py',
    source_dir='.',                       # train.py (and any helpers) ship together
    role=role,
    instance_type='ml.g5.2xlarge',
    instance_count=1,
    transformers_version='4.49', pytorch_version='2.5.1', py_version='py311',  # match a current DLC
    hyperparameters={'epochs': 1, 'lr': 2e-4, 'lora_r': 16, 'lora_alpha': 32},
)

estimator.fit()                           # or estimator.fit({'train': 's3://.../order-data/'})
print('model artifact:', estimator.model_data)   # s3://.../model.tar.gz

### B3 — register in the Model Registry

`register(...)` creates a versioned entry in a **Model Package Group** — SageMaker's
answer to Kubeflow Hub. A package starts `PendingManualApproval`; an eval gate
(Lab 3's numbers, run as a SageMaker Pipelines step) flips it to `Approved`, and
only approved packages deploy. That gate is the same idea as the README's
"regression blocks the deploy".

In [ ]:
model_package = estimator.register(
    model_package_group_name='qwen-order-parser',
    content_types=['application/json'],
    response_types=['application/json'],
    inference_instances=['ml.g5.xlarge'],
    transform_instances=['ml.g5.xlarge'],
    approval_status='PendingManualApproval',
)
print('registered:', model_package.model_package_arn)

### B4 — deploy an Endpoint and invoke

`deploy(...)` stands up a real-time **SageMaker Endpoint** (the managed twin of a
KServe `InferenceService`). Send the same system+user turns and watch it return
strict JSON. **Delete the endpoint when done** — it bills per hour.

In [ ]:
predictor = estimator.deploy(initial_instance_count=1, instance_type='ml.g5.xlarge')

resp = predictor.predict({'inputs': [
    {'role': 'system', 'content': 'You are an order-intent parser. Reply with ONLY a JSON object.'},
    {'role': 'user',   'content': 'can I get 3 lavender shampoo shipped to Hanoi'},
]})
print(resp)        # -> {"intent":"order","item":"lavender shampoo","qty":3,"city":"Hanoi"}

predictor.delete_endpoint()    # stop the meter

---
## The one lesson

The **training code is identical** — `train.py`, the same QLoRA on the same 3B
model from Lab 2. What a platform adds is everything *around* the training:

- a **launch idiom** — Kubeflow takes a *function* (`CustomTrainer`), SageMaker
  takes a *script* (`entry_point`); both run the same body on managed GPUs.
- a **registry** — Kubeflow Hub / SageMaker Model Registry — so the artifact is
  versioned, not a file on a laptop.
- an **eval gate** before deploy, and a **serving** surface (KServe / Endpoint).

Kaggle Lab 2 taught you *what* fine-tuning does. This lab is *how the same step
lives inside a platform* — on your own cluster (Kubeflow) or fully managed
(SageMaker).